# レース当日に当日配布データを取得して実行する例
 - 当日配布データからタイムテーブルを取得
 - 今回は8/30の例

## Import

In [ ]:
import requests
import pandas as pd
import joblib

import time
from datetime import datetime, timedelta

## Load Model and Read Data

In [ ]:
loaded_model = joblib.load("./model/decision_tree_model_20250821.pkl")

col_list = [
    "race_id", "year", "month", "day", "times",
    "place", "daily", "race_num", "horse", "jockey_id",
    "horse_N", "waku_num", "horse_num", "class_code", "track_code",
    "corner_num", "dist", "state", "weather", "age_code",
    "sex", "age", "basis_weight", "blinker", "weight",
    "inc_dec", "weight_code", "win_odds", "rank", "time_diff",
    "time", "corner1_rank", "corner2_rank", "corner3_rank", "corner4_rank",
    "last_3F_time", "last_3F_rank", "Ave_3F", "PCI", "last_3F_time_diff",
    "leg", "pop", "prize", "error_code",
    "father", "mother", "id"
]
dat_2024 = pd.read_csv("./input_data/record_data_2024.csv", encoding="shift_jis", names=col_list, low_memory=False)
dat_2025 = pd.read_csv("./input_data/record_data_2025_20250830.csv", encoding="shift_jis", names=col_list, low_memory=False)

## Get Data
 当日配布データのAPIを利用してデータを取得.

In [ ]:
url = "https://172.192.40.114/data"

headers = {
    # 認証キー:当日データAPIを使う時は共通
    "api-key": "AI_Keiba_2025",
    # "Racecards" or "Odds"
    "type_of_data": "Racecards",
    # Racecardsのときは日付, OddsのときはYYYYMMDDJJRRを入力:JJは場所コード, RRはレース番号
    # 場所コード："札幌", "函館", "福島", "新潟", "東京", "中山", "中京", "京都", "阪神", "小倉"は順番に01, 02, ..., 10
    "id": "20250830"
}

# --- リクエスト送信 ---
# 形だけ証明書なのでverify=Falseにする: Warningは出るが無視でオッケー
response = requests.get(url, headers=headers, verify=False)
res_data = response.json()

C:\Users\owner\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1020: InsecureRequestWarning: Unverified HTTPS request is being made to host '172.192.40.114'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


In [ ]:
timetable = pd.DataFrame(res_data["data"]["timetable"])
runtable = pd.DataFrame(res_data["data"]["runtable"])

In [ ]:
timetable

,year,month,day,place,race_num,start_time,class_code,track_code
0,2025,8,30,札幌,1,09:50,7,17
1,2025,8,30,札幌,2,10:25,7,17
2,2025,8,30,札幌,3,10:55,7,24
3,2025,8,30,札幌,4,11:25,7,17
4,2025,8,30,札幌,5,12:15,15,24
5,2025,8,30,札幌,6,12:45,7,17
6,2025,8,30,札幌,7,13:15,23,24
7,2025,8,30,札幌,8,13:45,23,24
8,2025,8,30,札幌,9,14:15,131,17
9,2025,8,30,札幌,10,14:50,43,24


In [ ]:
runtable.head()

,place,race_num,horse_num,dist,horse,sex,age,jockey,loaf_weight,father,mother,id,waku_num,race_id
0,札幌,1,1,1500,リバテイー,牝,2,横山武史,55.0,ヴァンキッシュラン,ウィッシングタイム,23105932,1,202508300102030101
1,札幌,1,2,1500,エリカビアリッツ,牝,2,佐々木大,55.0,キズナ,ブレーヴアンナ,23102608,2,202508300102030102
2,札幌,1,3,1500,ノアールビーナス,牝,2,川又賢治,55.0,ドレフォン,カフジビーナス,23104047,3,202508300102030103
3,札幌,1,4,1500,コマチチャン,牝,2,斎藤新,55.0,キズナ,シュウギン,23106086,4,202508300102030104
4,札幌,1,5,1500,ベンテンカグラ,牝,2,石川倭,55.0,アドマイヤムーン,スガノグラスワン,23107419,5,202508300102030105


## Creating input data for prediction

In [ ]:
# idを10桁にする
runtable["id"] = runtable["id"] + 2000000000

In [ ]:
# データを縦積み
dat = pd.concat([dat_2024, dat_2025, runtable], axis=0)

# 馬ごとに時系列順に並べ替え
features = dat.sort_values(["id", "race_id"]).copy()
# 馬ごとに前レース結果を付与
features["rank_last"] = features.groupby("id")[["rank"]].shift(1)
# 馬ごとにデータ上の最初のレースは前レース結果が無いので0埋め：本来は新馬戦のみ欠損となるべき
features["rank_last"] = features["rank_last"].fillna(0)

print(runtable.shape)
test = pd.merge(runtable, features[["id", "race_id", "rank_last"]], how="inner", on=["id", "race_id"])

# 各API用のrace_idを作成
test["race_id_odds"] = test["race_id"].apply(lambda x: int(str(x)[0:10] + str(x)[14:16]))
test["race_id_vote"] = test["race_id"].apply(lambda x: int(str(x)[0:4] + str(x)[8:16]))

# 馬番号を0埋め2桁にする
test["comb"] = test["horse_num"].apply(lambda x: str(x).zfill(2))
print(test.shape)

(488, 14)
(488, 18)


In [ ]:
test[["age", "rank_last", "race_id", "race_id_odds", "race_id_vote", "comb"]].tail(20)

,age,rank_last,race_id,race_id_odds,race_id_vote,comb
468,4,1.0,202508300704031109,202508300711,202507040311,09
469,4,1.0,202508300704031110,202508300711,202507040311,10
470,4,1.0,202508300704031111,202508300711,202507040311,11
471,9,4.0,202508300704031112,202508300711,202507040311,12
472,5,11.0,202508300704031113,202508300711,202507040311,13
473,5,15.0,202508300704031114,202508300711,202507040311,14
474,6,5.0,202508300704031115,202508300711,202507040311,15
475,4,1.0,202508300704031116,202508300711,202507040311,16
476,3,7.0,202508300704031201,202508300712,202507040312,01
477,3,5.0,202508300704031202,202508300712,202507040312,02


## Predict

In [ ]:
# 予測確率の付与
test["pred_prob"] = loaded_model.predict_proba(test[["age", "rank_last"]])[:, 1]
test[["horse_num", "horse", "age", "rank_last", "race_id", "pred_prob", "comb"]].head(20)

,horse_num,horse,age,rank_last,race_id,pred_prob,comb
0,1,リバテイー,2,4.0,202508300102030101,0.069741,01
1,2,エリカビアリッツ,2,3.0,202508300102030102,0.202695,02
2,3,ノアールビーナス,2,8.0,202508300102030103,0.028463,03
3,4,コマチチャン,2,10.0,202508300102030104,0.021053,04
4,5,ベンテンカグラ,2,11.0,202508300102030105,0.009368,05
5,6,アスコットダンス,2,5.0,202508300102030106,0.069741,06
6,7,ラスティングスノー,2,2.0,202508300102030107,0.202695,07
7,8,クラリスヒメ,2,7.0,202508300102030108,0.028463,08
8,1,ショウナンバーボン,2,3.0,202508300102030201,0.202695,01
9,2,ロイヤルタイム,2,6.0,202508300102030202,0.063275,02


## Running and Vote
タイムテーブルに合わせて, 5分前オッズを取得して投票するサンプル.
 - 今回は, レースごとに1着確率が高い馬に対して5分前オッズを使用してベット額を決めて, 投票をする.

In [ ]:
##### netkeibaのアカウント情報
# 皆さんのアカウント情報に書き換えて下さい
login_id = ""
password = ""

In [ ]:
# 最初はポイントを設定する：投票するとレスポンスに残ポイント情報があるので後はそこで更新
remaining_points = 100 * 10000

In [ ]:
def vote_api_login_fun(login_id, password):
    url = "https://masters.netkeiba.com/ai2025_student/api/login"
    headers = {
        "Content-Type": "application/json"
    }
    payload = {
        "login_id": login_id,
        "password": password
    }

    res = requests.post(url, headers=headers, json=payload)
    if res.status_code == 200:
        print("ログイン成功")
        return res.json()["data"]["access_token"]
    else:
        print(f"ログイン失敗: {res.status_code}")
        return {}

def vote_api_logout_fun(access_token):
    url = "https://masters.netkeiba.com/ai2025_student/api/logout"
    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    res = requests.post(url, headers=headers)
    if res.status_code == 200:
        print("ログアウト成功")
    else:
        print(f"ログアウト失敗: {response.status_code}")

def vote_api_bet_fun(bet_data_json, access_token):
    url = "https://masters.netkeiba.com/ai2025_student/api/bet"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}"
    }
    payload = {
        "bet_data": [bet_data_json]
    }
    # POSTリクエスト送信
    res = requests.post(url, headers=headers, json=payload)

    # レスポンス確認
    if res.status_code == 200:
        print("ベット成功")
        return res.json()
    else:
        print(f"エラー: {res.status_code}")
        print(res.text)
        return {}

def get_odds_rt_fun(race_id):
    url = "https://172.192.40.114/data"
    headers = {
        # 認証キー:当日データAPIを使う時は共通
        "api-key": "AI_Keiba_2025",
        # "Racecards" or "Odds"
        "type_of_data": "Odds",
        # Racecardsのときは日付, OddsのときはYYYYMMDDJJRRを入力:JJは場所コード, RRはレース番号
        # 場所コード："札幌", "函館", "福島", "新潟", "東京", "中山", "中京", "京都", "阪神", "小倉"は順番に01, 02, ..., 10
        "id": race_id
    }

    # 形だけ証明書なのでverify=Falseにする: Warningは出るが無視でオッケー
    res = requests.get(url, headers=headers, verify=False)

    # --- 結果表示 ---
    if res.status_code == 200:
        data = res.json()
        print("成功:", data["message"])
        return data
    else:
        print(f"エラー: {res.status_code}")
        print(res.text)
        return {}

In [ ]:
timetable = timetable.sort_values("start_time")
for _, rows in timetable.iterrows():

    ###### 待機
    # 対象レース
    place = rows.place
    race_num = rows.race_num
    start_time = rows.start_time

    race_id_odds = str((test[(test["place"]==place) & (test["race_num"]==race_num)]["race_id_odds"].values[0]))
    race_id_vote = str((test[(test["place"]==place) & (test["race_num"]==race_num)]["race_id_vote"].values[0]))

    target_time = datetime.strptime(start_time, "%H:%M").replace(
        year=datetime.now().year,
        month=datetime.now().month,
        day=datetime.now().day
    )

    # 現在時刻を取得
    now = datetime.now()

    # すでに過ぎていたらスキップ
    if now > target_time:
        print(f"[{start_time}] → すでに時刻を過ぎています。スキップ。")
        continue

    # 残り秒数を計算
    wait_seconds = (target_time - now).total_seconds()

    # 5分前オッズを発走時刻の4分10秒前に取得
    wait_seconds = wait_seconds - (4 * 60 + 10)

    # 5分前オッズがすでに過ぎていたらスキップ
    if wait_seconds < 0:
        print(f"5分前オッズ取得設定時刻を過ぎています")
        continue

    print(f"{start_time}")
    print(f"5分前オッズ取得まで {int(wait_seconds)} 秒待機...")

    # 待機
    time.sleep(wait_seconds)


    ###### 投票ポイント計算
    # 5分前オッズ取得
    odds_rt = pd.DataFrame(get_odds_rt_fun(race_id_odds)["data"]["odds_rt"])

    # 単勝オッズのある中から確率最大のものを抽出
    odds_rt_win = odds_rt[odds_rt["odds_type"]==1]
    bet_data = pd.merge(
        test[["place", "race_num", "race_id_odds", "comb", "pred_prob"]],
        odds_rt_win[["race_id", "comb", "odds"]], how="inner",
        left_on=["race_id_odds", "comb"], right_on=["race_id", "comb"]
    )
    bet_data = bet_data.loc[bet_data.groupby(["place", "race_num"])['pred_prob'].idxmax()].copy()
    bet_horse_num_str = str(int(bet_data["comb"].values[0]))
    bet_odds_rt = int(bet_data["odds"].values[0])

    # オッズで割ったものの10%をベット：サンプルロジック
    # 100ポイント単位にするのを忘れない
    bet_amount = int(((remaining_points / bet_odds_rt / 10) // 100) * 100)
    bet_data_json = {
        # race_id
        "race_id": race_id_vote,
        # markは必須なので形式的にベットする馬を指定
        "mark": {bet_horse_num_str:1},
        # ベット
        "bet": [{"bet_id": f"b1_c0_{bet_horse_num_str}", "money": str(bet_amount)}]
    }

    ###### 投票
    # ログイン
    access_token = vote_api_login_fun(login_id, password)

    # ベット
    res = vote_api_bet_fun(bet_data_json, access_token)

    if len(res) > 0:
        # 残ポイントの更新
        remaining_points = int(res["remaining_money"])
        print(res)

    # ログアウト
    vote_api_logout_fun(access_token)

[09:50] → すでに時刻を過ぎています。スキップ。
[10:01] → すでに時刻を過ぎています。スキップ。
[10:10] → すでに時刻を過ぎています。スキップ。
[10:25] → すでに時刻を過ぎています。スキップ。
[10:35] → すでに時刻を過ぎています。スキップ。
[10:45] → すでに時刻を過ぎています。スキップ。
[10:55] → すでに時刻を過ぎています。スキップ。
[11:05] → すでに時刻を過ぎています。スキップ。
[11:15] → すでに時刻を過ぎています。スキップ。
[11:25] → すでに時刻を過ぎています。スキップ。
[11:35] → すでに時刻を過ぎています。スキップ。
[11:45] → すでに時刻を過ぎています。スキップ。
[12:15] → すでに時刻を過ぎています。スキップ。
[12:25] → すでに時刻を過ぎています。スキップ。
[12:35] → すでに時刻を過ぎています。スキップ。
[12:45] → すでに時刻を過ぎています。スキップ。
[12:55] → すでに時刻を過ぎています。スキップ。
[13:05] → すでに時刻を過ぎています。スキップ。
[13:15] → すでに時刻を過ぎています。スキップ。
[13:25] → すでに時刻を過ぎています。スキップ。
[13:35] → すでに時刻を過ぎています。スキップ。
[13:45] → すでに時刻を過ぎています。スキップ。
[13:55] → すでに時刻を過ぎています。スキップ。
[14:05] → すでに時刻を過ぎています。スキップ。
[14:15] → すでに時刻を過ぎています。スキップ。
[14:25] → すでに時刻を過ぎています。スキップ。
[14:35] → すでに時刻を過ぎています。スキップ。
[14:50] → すでに時刻を過ぎています。スキップ。
[15:01] → すでに時刻を過ぎています。スキップ。
[15:10] → すでに時刻を過ぎています。スキップ。
5分前オッズ取得設定時刻を過ぎています
15:35
5分前オッズ取得まで 392 秒待機...


C:\Users\owner\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1020: InsecureRequestWarning: Unverified HTTPS request is being made to host '172.192.40.114'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


成功: OK
ログイン成功
ベット成功
{'status': 'OK', 'data': {'list_bet_race': ['中京11R 名古屋城Ｓ：合計33,300円 単勝 1点'], 'success_count': 1, 'error_count': 0}, 'remaining_money': '933400'}
ログアウト成功
15:45
5分前オッズ取得まで 599 秒待機...
